# SoundStream Demo

This notebook demonstrates the main purpose of this repo - trained SoundStream neural audio codec. It takes any audio URL, passes it through the codec and plays the original vs reconstructed audio to understand quality of model

My implementation of this model has `STOI` = 0.8229399738928121 and `NISQA` = 1.9275813531784611 on `test-clean` of LibriSpeec.

## 1. Getting code

In [1]:
!git clone https://github.com/Vdv09/Neural-Audio-Codec.git

%cd Neural-Audio-Codec

!pip install -r requirements.txt

Cloning into 'Neural-Audio-Codec'...
remote: Enumerating objects: 156, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 156 (delta 39), reused 151 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (156/156), 54.27 KiB | 3.88 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/Neural-Audio-Codec
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.6/796.6 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.

## 2. Download best checkpoint

In [2]:
import os

os.makedirs("checkpoints", exist_ok=True)

!gdown 13trOe0Fj-IwJxkHFumprQND0lOvKe9G5 -O checkpoints/model_best.pth

Downloading...
From (original): https://drive.google.com/uc?id=13trOe0Fj-IwJxkHFumprQND0lOvKe9G5
From (redirected): https://drive.google.com/uc?id=13trOe0Fj-IwJxkHFumprQND0lOvKe9G5&confirm=t&uuid=2f077a84-7015-4f81-90b1-9156c429e495
To: /content/Neural-Audio-Codec/checkpoints/model_best.pth
100% 335M/335M [00:09<00:00, 36.9MB/s]


## 3. Load model

In [3]:
import torch
from src.model import SoundStream

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

ckpt = torch.load("checkpoints/model_best.pth", map_location=device, weights_only=False)

model = SoundStream().to(device)
model.load_state_dict(ckpt["model"])
model.eval()

print(f"Loaded checkpoint from step {ckpt['step']}")

Using device: cpu
Loaded checkpoint from step 10000


## 4. Model inference

Notice that you can use with any url for audio file

In [4]:
import urllib.request
import torchaudio
import torch
import numpy as np
from IPython.display import display, Audio

SAMPLE_RATE = 16000

AUDIO_URL = "https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav"

urllib.request.urlretrieve(AUDIO_URL, "input.wav")

audio, sr = torchaudio.load("input.wav")

if sr != SAMPLE_RATE:
    audio = torchaudio.functional.resample(audio, sr, SAMPLE_RATE)

audio = audio[:1]

with torch.no_grad():
    out = model(audio.unsqueeze(0).to(device))

reconstructed = out["audio_hat"].squeeze().cpu()

print("Original:")
display(Audio(audio.numpy(), rate=SAMPLE_RATE))

print("Reconstructed:")
display(Audio(reconstructed.numpy(), rate=SAMPLE_RATE))

Original:


Reconstructed:


### Metrics

You can also make sure that our metrics are good

First, let's load data

In [5]:
import os


os.makedirs("data", exist_ok=True)

!wget -q https://www.openslr.org/resources/12/test-clean.tar.gz -O /tmp/test-clean.tar.gz
!tar -xzf /tmp/test-clean.tar.gz -C data/

In [ ]:
!python evaluate.py \
  --checkpoint checkpoints/model_best.pth \
  --data_dir data/LibriSpeech/test-clean

  0% 0/2620 [00:00<?, ?it/s]downloading https://github.com/gabrielmittag/NISQA/raw/refs/heads/master/weights/nisqa.tar to /root/.torchmetrics/NISQA/nisqa.tar
  1% 19/2620 [01:18<2:54:08,  4.02s/it]